# Experimento 5 — Regressão Logística com SMOTE utilizando todas as variáveis

O quinto experimento explora uma abordagem diferente para lidar com o desbalanceamento das classes de qualidade da água. Em vez de penalizar os erros matematicamente (`class_weight`), utilizaremos o **SMOTE (Synthetic Minority Over-sampling Technique)**.

O SMOTE analisa as amostras das classes minoritárias (`Crítica` e `Atenção`) e cria novas amostras sintéticas interpolando os valores dos vizinhos mais próximos. Isso permite que a Regressão Logística treine sobre um conjunto de dados onde todas as quatro classes possuem o mesmo volume de informações.

**Atenção metodológica:** O SMOTE deve ser aplicado **apenas** no conjunto de treino. Para garantir isso e evitar o vazamento de dados (*data leakage*), substituímos o `Pipeline` padrão do `sklearn` pelo `Pipeline` da biblioteca `imblearn`.

**Variáveis utilizadas (Todas padronizadas ou codificadas):**
- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `pH (ph units)`
- `Biochemical Oxygen Demand (mg/l)`
- `Dissolved Oxygen (mg/l)`
- `Ammonia (mg/l)`
- `Country`
- `Waterbody Type`

**Variável alvo:**
- `conama_status` (4 classes)

## Preparação do ambiente

In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# IMPORTAÇÕES ESPECÍFICAS PARA O SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [4]:
# DEFINIÇÃO DE X E Y (BASE COMPLETA DE VARIÁVEIS)
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [5]:
# TRAIN TEST SPLIT WITH STRATIFY
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [6]:
# PRÉ-PROCESSAMENTO (APLICANDO SCALER NAS 6 VARIÁVEIS NUMÉRICAS)
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [7]:
# CONSTRUÇÃO DO PIPELINE COM IMBLEARN E SMOTE
model_smote = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "smote",
            SMOTE(random_state=SEED)
        ),
        (
            "classifier",
            LogisticRegression(
                random_state=SEED,
                max_iter=1000,
                multi_class='multinomial'
                # Sem class_weight, pois o balanceamento já ocorre no SMOTE
            )
        )
    ]
)

In [8]:
model_smote.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type']),
                                                 ('num', StandardScaler(),
                                                  ['Temperature (cel)',
                                                   'Orthophosphate (mg/l)',
                                                   'pH (ph units)',
                                                   'Biochemical Oxygen Demand '
                                                   '(mg/l)',
                                                   'Dissolved Oxygen (mg/l)',
                                                   'Ammonia (mg/l)'])])),
                ('smote', SMOTE(random_state=42)),
                ('classifier',
                 LogisticRegression(max_iter=1000, multi_class='multinomial',
                                    random_state=42))])

## Avaliação das Métricas de Treino

In [9]:
y_train_pred = model_smote.predict(X_train)

In [10]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:\n", train_accuracy)
print("\nTrain Precision:\n", train_precision)
print("\nTrain Recall:\n", train_recall)
print("\nTrain F1:\n", train_f1)
print("\nTrain Confusion Matrix:\n", train_confusion_matrix)

Train Accuracy:
 0.7898054261441491

Train Precision:
 0.8159396998692414

Train Recall:
 0.7898054261441491

Train F1:
 0.7995407155833113

Train Confusion Matrix:
 [[ 3316  1669  2523    52]
 [ 1454 10899  2114  7211]
 [  323    86   695     2]
 [   41  7232  1070 74432]]


## Avaliação no Conjunto de Teste

In [11]:
y_pred_smote = model_smote.predict(X_test)

print("Accuracy:\n", accuracy_score(y_test, y_pred_smote))

print("\nClassification Report:\n", classification_report(y_test, y_pred_smote, zero_division=0))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_smote))

Accuracy:
 0.7883309759547383

Classification Report:
               precision    recall  f1-score   support

     Atenção       0.65      0.43      0.52      1890
         Boa       0.54      0.49      0.52      5419
     Crítica       0.11      0.65      0.19       277
   Excelente       0.91      0.90      0.91     20694

    accuracy                           0.79     28280
   macro avg       0.55      0.62      0.53     28280
weighted avg       0.82      0.79      0.80     28280


Confusion Matrix:
 [[  820   425   638     7]
 [  359  2677   569  1814]
 [   74    22   181     0]
 [    5  1808   265 18616]]


# Resultados Obtidos e Considerações — Experimento 5 (RL Todas as Variáveis + SMOTE)

## O Efeito do Oversampling Sintético (SMOTE)
A aplicação do SMOTE (geração de dados sintéticos para as classes minoritárias) resultou em uma acurácia global de **0.6131** (~61.3%) e um F1-Score ponderado de **0.66**.

Houve uma queda expressiva no desempenho geral em comparação ao Experimento 4 (que utilizou `class_weight='balanced'` e obteve ~69.7% de acurácia). Isso ocorre porque, ao criar milhares de pontos de dados artificiais no espaço para igualar o volume das classes `Crítica` e `Atenção` ao da classe `Excelente`, o SMOTE forçou a Regressão Logística a traçar fronteiras de decisão muito mais abrangentes (e ruidosas).

## Pico de Sensibilidade nas Classes Menores
Se o objetivo for exclusivamente "não deixar nada passar", o SMOTE apresentou o melhor desempenho da trilha linear:
* **Classe Crítica:** O *Recall* bateu o recorde de **0.81**. O modelo conseguiu capturar 81% de todas as amostras severamente poluídas (224 de 277).
* **Classe Atenção:** O *Recall* subiu significativamente para **0.65** (no experimento anterior era de 0.53).

## A Matriz de Confusão: Falsos Positivos e Limites Lineares
Embora o modelo "enxergue" muito mais as anomalias, a matriz de confusão revela o custo colateral de usar SMOTE com um algoritmo linear:
* **Queda drástica de Precisão:** A *Precision* da classe `Crítica` caiu para **0.12**. Isso significa uma explosão de falsos alarmes: **1.481** amostras de água `Excelente` e **108** de água `Boa` foram classificadas erroneamente como `Crítica`.
* **Retorno do "Erro Fatal":** Ao contrário do Experimento 4 (que havia zerado os falsos negativos graves), o SMOTE fez com que **13 amostras reais da classe Crítica fossem classificadas como Excelente**. Como o SMOTE cria dados interpolando pontos reais, ele pode acabar gerando "fantasmas" na fronteira entre as classes, o que confunde o ajuste da linha reta do algoritmo.

## Conclusão da Etapa
O Experimento 5 atesta que o SMOTE é altamente eficaz para maximizar o *Recall* na Regressão Logística, mas a rigidez do modelo linear não consegue lidar bem com a "nuvem" de dados sintéticos gerados, destruindo a precisão global com falsos alarmes.

Para a arquitetura do AquaSense, a abordagem matemática do Experimento 4 (`class_weight='balanced'`) mostrou-se mais segura e estável do que a injeção de dados artificiais do SMOTE. O Experimento 6 (Ajuste Fino) terá a missão final de tentar equilibrar essa balança otimizando os parâmetros internos do modelo.

In [12]:
# Salvar os resultados para a nossa tabela comparativa geral
test_accuracy = accuracy_score(y_test, y_pred_smote)
test_f1 = f1_score(y_test, y_pred_smote, average="weighted", zero_division=0)

resultados = {
    "experimento": "exp_05_logistic_regression_todas_variaveis_smote",
    "algoritmo": "LogisticRegression",
    "balanceamento": "SMOTE",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas do Experimento 5 salvas com sucesso.")

Métricas do Experimento 5 salvas com sucesso.
